In [ ]:
# Install Cadabra if necessary, and enable Cadabra input in this notebook.
try:
    import cadabra2
except ImportError:
    %pip install -q cadabra2
%load_ext cadabra2_jupyter

# The Kerr spacetime

This notebook computes the Christoffel symbols, the Riemann tensor, the Ricci
tensor and the Kretschmann scalar of the Kerr metric, in Boyer-Lindquist
coordinates. It follows the same steps as the Schwarzschild example, with one
change in how the components are simplified, explained below.

We start by declaring the coordinates, the indices and the metric.

In [ ]:
__cdbkernel__.display_fractions=True

In [ ]:
{t,r,\theta,\phi}::Coordinate;
{\mu,\nu,\rho,\sigma,\lambda,\kappa,\chi,\gamma}::Indices(values={t,r,\theta,\phi}, position=fixed);
\partial{#}::PartialDerivative;
g_{\mu\nu}::Metric.
g^{\mu\nu}::InverseMetric.

For the evaluation of components the scalar backend (Sympy) needs to see all
dependence on the coordinates explicitly. Rather than introducing the usual
abbreviations $\Sigma = r^2 + a^2\cos^2\theta$ and $\Delta = r^2 - 2Mr + a^2$
as abstract functions, we therefore write the metric components out in full.

In [ ]:
ss:= { g_{t t} = -(1-2 M r/(r**2+a**2 \cos(\theta)**2)),
       g_{t \phi} = -2 M a r \sin(\theta)**2/(r**2+a**2 \cos(\theta)**2),
       g_{\phi t} = -2 M a r \sin(\theta)**2/(r**2+a**2 \cos(\theta)**2),
       g_{r r} = (r**2+a**2 \cos(\theta)**2)/(r**2-2 M r+a**2),
       g_{\theta\theta} = r**2+a**2 \cos(\theta)**2,
       g_{\phi\phi} = (r**2+a**2
                       +2 M a**2 r \sin(\theta)**2/(r**2+a**2 \cos(\theta)**2))
                      \sin(\theta)**2 }.

By default, [`evaluate`](https://cadabra.science/manual/evaluate.html) runs Sympy's `simplify` on every component
after every intermediate step. That works well for Schwarzschild, but for Kerr
it is very slow, and its running time varies a lot from run to run (Sympy's
multivariate factorisation picks random evaluation points, and some choices
are very expensive).

Since all components are rational functions of $r$, $\cos\theta$ and
$\sin\theta$, there is a cheaper and fully reliable alternative: the
`trigrat` function from the `cdb.core.trigrat` package brings such
expressions into a canonical form, so that a component which vanishes is always
recognised as zero, and removes the vanishing components.

In [ ]:
from cdb.core.trigrat import trigrat

We can now compute the inverse metric with [`complete`](https://cadabra.science/manual/complete.html), and bring it into
canonical form.

In [ ]:
complete(ss, $g^{\mu\nu}$)
trigrat(ss);

For the Christoffel symbols we switch off the built-in simplification with
`simplify=False`, and apply `trigrat` to the result afterwards.

In [ ]:
ch:= \Gamma^{\mu}_{\nu\rho} = 1/2 g^{\mu\sigma} (
                                   \partial_{\rho}{g_{\nu\sigma}}
                                  +\partial_{\nu}{g_{\rho\sigma}}
                                  -\partial_{\sigma}{g_{\nu\rho}} ):

evaluate(ch, ss, rhsonly=True, simplify=False)
trigrat(ch);

The Riemann tensor follows in the same way. There are 88 non-vanishing
components $R^{\rho}{}_{\sigma\mu\nu}$ (counting both orders of the last pair
of indices).

In [ ]:
rm:= R^{\rho}_{\sigma\mu\nu} = \partial_{\mu}{\Gamma^{\rho}_{\nu\sigma}}
                                  -\partial_{\nu}{\Gamma^{\rho}_{\mu\sigma}}
                                  +\Gamma^{\rho}_{\mu\lambda} \Gamma^{\lambda}_{\nu\sigma}
                                  -\Gamma^{\rho}_{\nu\lambda} \Gamma^{\lambda}_{\mu\sigma}:
substitute(rm, ch)
evaluate(rm, ss, rhsonly=True, simplify=False)
trigrat(rm);

The Kerr metric is a vacuum solution, so the Ricci tensor has to vanish. All
of its components are indeed recognised as zero:

In [ ]:
rc:= R_{\sigma\nu} = R^{\rho}_{\sigma\rho\nu}:
substitute(rc, rm)
evaluate(rc, ss, rhsonly=True, simplify=False)
trigrat(rc);

Finally, the Kretschmann scalar.

In [ ]:
K:= K = R^{\mu}_{\nu\rho\sigma} R^{\lambda}_{\kappa\gamma\chi}
           g_{\mu\lambda} g^{\nu\kappa} g^{\rho\gamma} g^{\sigma\chi}:
substitute(K, rm)
evaluate(K, ss, rhsonly=True, simplify=False)
trigrat(K);

This is the known result

$$
K = \frac{48 M^2 \left(r^2 - a^2\cos^2\theta\right)
          \left(r^4 - 14 a^2 r^2\cos^2\theta + a^4\cos^4\theta\right)}
         {\left(r^2 + a^2\cos^2\theta\right)^6}\,,
$$

in factorised form. It is finite at the horizons, and diverges only where
$r^2 + a^2\cos^2\theta = 0$, that is at $r=0$, $\theta=\pi/2$: the ring
singularity. For $a=0$ it reduces to the Schwarzschild value:

In [ ]:
K0:= @(K):
substitute(K0, $a=0$)
trigrat(K0);